# 8-Queens CSP: Backtracking 


This is the initialization for the basic function 


In [ ]:
N = 8
vars = [f"Q{i}" for i in range(1, N+1)]

def ok(assign, var, val):
    i = int(var[1:])
    for v, val2 in assign.items():
        j = int(v[1:])
        if val2 == val or abs(val2 - val) == abs(i - j):
            return False
    return True

def show(sol):
    rows = [sol[f"Q{c}"] for c in range(1, N+1)]
    print(rows)
    for r in range(1, N+1):
        print(" ".join("Q" if rows[c-1]==r else "." for c in range(1, N+1)))

: 

## 1. Plain Backtracking

In [ ]:
def backtrack(assign, steps):
    if len(assign) == N:
        return dict(assign)
    var = vars[len(assign)]
    for val in range(1, N+1):
        steps[0] += 1
        if ok(assign, var, val):
            assign[var] = val
            result = backtrack(assign, steps)
            if result:
                return result
            del assign[var]
    return None

steps1 = [0]
sol1 = backtrack({}, steps1)
print("Steps:", steps1[0])
show(sol1)

## 2. Backtracking + MRV + LCV

In [ ]:
def mrv(assign, domains):
    unassigned = [v for v in vars if v not in assign]
    return min(unassigned, key=lambda v: len(domains[v]))

def lcv(var, assign, domains):
    i = int(var[1:])
    def conflicts(val):
        c = 0
        for other in vars:
            if other == var or other in assign:
                continue
            j = int(other[1:])
            c += sum(1 for v2 in domains[other] if v2==val or abs(v2-val)==abs(i-j))
        return c
    return sorted(domains[var], key=conflicts)

def forward_check(var, val, assign, domains):
    i = int(var[1:])
    pruned = {}
    for other in vars:
        if other == var or other in assign:
            continue
        j = int(other[1:])
        removed = [v2 for v2 in domains[other] if v2==val or abs(v2-val)==abs(i-j)]
        for r in removed:
            domains[other].remove(r)
        if removed:
            pruned[other] = removed
        if not domains[other]:
            return pruned, True
    return pruned, False

def restore(pruned, domains):
    for var, vals in pruned.items():
        domains[var].extend(vals)

def backtrack_mrv_lcv(assign, domains, steps, log=True):
    if len(assign) == N:
        return dict(assign)
    var = mrv(assign, domains)
    values = lcv(var, assign, domains)
    if log:
        print(f"MRV picks {var} | LCV order: {values}")
    for val in values:
        steps[0] += 1
        if ok(assign, var, val):
            assign[var] = val
            pruned, fail = forward_check(var, val, assign, domains)
            if not fail:
                result = backtrack_mrv_lcv(assign, domains, steps, log)
                if result:
                    return result
            restore(pruned, domains)
            del assign[var]
    return None

domains2 = {v: list(range(1, N+1)) for v in vars}
steps2 = [0]
sol2 = backtrack_mrv_lcv({}, domains2, steps2)
print("Steps:", steps2[0])
show(sol2)

## 3. Backtracking + MRV + LCV + AC-3

In [ ]:
from collections import deque
import copy

def revise(domains, xi, xj):
    i, j = int(xi[1:]), int(xj[1:])
    changed = False
    for x in domains[xi][:]:
        if not any(y != x and abs(x-y) != abs(i-j) for y in domains[xj]):
            domains[xi].remove(x)
            changed = True
    return changed

def ac3(domains, arcs, log=False):
    q = deque(arcs)
    while q:
        xi, xj = q.popleft()
        if revise(domains, xi, xj):
            if log:
                print(f"  AC-3 shrinks {xi} -> {domains[xi]}")
            if not domains[xi]:
                return False
            for xk in vars:
                if xk != xi and xk != xj:
                    q.append((xk, xi))
    return True

def backtrack_ac3(assign, domains, steps, log=True):
    if len(assign) == N:
        return dict(assign)
    var = mrv(assign, domains)
    values = lcv(var, assign, domains)
    if log:
        print(f"MRV picks {var} | LCV order: {values}")
    for val in values:
        steps[0] += 1
        if ok(assign, var, val):
            assign[var] = val
            saved = copy.deepcopy(domains)
            domains[var] = [val]
            arcs = [(xk, var) for xk in vars if xk not in assign]
            if ac3(domains, arcs, log):
                result = backtrack_ac3(assign, domains, steps, log)
                if result:
                    return result
            domains.clear()
            domains.update(saved)
            del assign[var]
    return None

domains3 = {v: list(range(1, N+1)) for v in vars}
print("Initial AC-3 pass:")
ac3(domains3, [(xi, xj) for xi in vars for xj in vars if xi != xj], log=True)

steps3 = [0]
sol3 = backtrack_ac3({}, domains3, steps3)
print("Steps:", steps3[0])
show(sol3)

## 4. Comparison

In [ ]:
import pandas as pd

pd.DataFrame({
    "Method": ["Plain Backtracking", "MRV + LCV", "MRV + LCV + AC-3"],
    "Solution Found": [sol1 is not None, sol2 is not None, sol3 is not None],
    "Steps": [steps1[0], steps2[0], steps3[0]]
})